# Службен весник — M0 zero-shot: **NLLB-200 distilled 600M**

`01_3` wrote `pairs_sonar_filtered.parquet`: 295,628 mk/sq pairs over 2,282 gazette issues,
2001–2025. This notebook measures what `facebook/nllb-200-distilled-600M` does on the **test split of that
corpus with no adaptation at all** — arm **M0** of the experimental matrix, the number every
fine-tuned arm is later compared against.

It trains nothing. It generates, scores, and writes one result file.

| § | |
|---|---|
| 0 | Setup |
| 1 | Configuration |
| 2 | Chart styling |
| 3 | Load the evaluation split |
| 4 | Model, tokenizer, and the `forced_bos_token_id` canary |
| 5 | Generate — sharded, checkpointed, resumable |
| 6 | General quality: SacreBLEU, chrF2++, COMET |
| 7 | Pattern fidelity: articles, paragraphs, dates, money, percentages, numerals |
| 8 | Results, per direction |
| 9 | Write and report |

## Five things about this evaluation that are easy to get wrong

1. **Both directions are reported separately and never averaged.** MK→SQ and SQ→MK do not
   improve symmetrically under adaptation, and a mean of the two hides exactly the finding
   the paper is after. Every table in §8 is indexed by direction.

2. **`forced_bos_token_id` is asserted before anything long runs.** Getting it wrong produces
   fluent, well-formed output in the *wrong language*, which BLEU scores as a catastrophe but
   which no assertion in the generation loop would otherwise catch. §4 checks the id, then
   generates three segments and measures the script of the output before §5 is allowed to run.

3. **No n-gram repetition blocking.** `no_repeat_ngram_size` is left unset on purpose. Legal
   text legitimately repeats — «во согласност со членот 5 став 1 од овој закон» is not a
   degenerate loop — and blocking it silently truncates correct translations.

4. **The split must be the chronological one.** `01_1` §7 assigned train/dev/test by
   `sha1(filename) % 100`, which scatters every year across all three splits and leaks
   formulaic gazette text between them. §3 refuses to report a publishable number off that
   split; it reads the chronological, issue-disjoint corpus instead and says loudly which one
   it got.

5. **A zero-shot number is a floor, not a diagnosis.** NLLB-200 saw `mkd_Cyrl` and `als_Latn`
   in training but almost certainly no Macedonian legal gazette. A low BLEU here is expected
   and is the point; §7 is what says *how* it fails — whether it drops article numbers and
   money values, or whether it merely paraphrases.


---
## 0. Setup

**Colab: Runtime → Change runtime type → a GPU.** A T4 is enough for the 600M model at `BATCH_SIZE = 32`; an A100 is roughly four
times faster and is worth it for the full test split. CPU works and is about fifty times slower —
use `MAX_EVAL_SEGMENTS = 200` if that is all you have.

Outside Colab the notebook reads the corpus from `data/processed/corpus/` and checkpoints to
`_local/`, so it runs unchanged on the local box.


In [ ]:
import importlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def _ensure(pip_name: str, module: str | None = None) -> None:
    try:
        importlib.import_module(module or pip_name)
    except ImportError:
        print(f"installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)


_ensure("sacrebleu")
_ensure("sentencepiece")
if IN_COLAB:
    _ensure("transformers")
    from google.colab import drive

    drive.mount("/content/drive")

print("running in colab:", IN_COLAB)

In [ ]:
import json
import re
import time
import unicodedata
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import sacrebleu
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 160)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE, "| sacrebleu", sacrebleu.__version__)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:,.0f} GB")

---
## 1. Configuration


### Configuration

One block, named constants, no hyperparameters anywhere else in the notebook. `RUN_NAME`
follows the project convention `{{base}}_{{arm}}_{{strategy}}_{{seed}}`; M0 uses beam search with
no sampling, so the seed is nominal and recorded only for symmetry with the trained arms.


In [ ]:
# ---- run identity -----------------------------------------------------------
BASE = "nllb600m"
ARM = "M0"
STRATEGY = "zeroshot"
SEED = 0
RUN_NAME = f"{BASE}_{ARM}_{STRATEGY}_s{SEED}"

# ---- model -------------------------------------------------------------------
MODEL_NAME = "facebook/nllb-200-distilled-600M"
# Both codes are already in the NLLB tokenizer and model. No vocabulary surgery,
# no embedding resize -- §4 asserts this rather than trusting it.
MK = "mkd_Cyrl"
SQ = "als_Latn"
DIRECTIONS = [("mk2sq", MK, SQ), ("sq2mk", SQ, MK)]

# ---- where the data lives ----------------------------------------------------
if IN_COLAB:
    DATA_ROOT = Path("/content")
    WORK_ROOT = Path("/content/drive/MyDrive/mksq")       # survives a dead runtime
else:
    DATA_ROOT = Path("../data/processed")
    WORK_ROOT = Path("../_local")

# The chronological, issue-disjoint corpus written by scripts/prepare_corpus.py.
CORPUS_DIR = DATA_ROOT / "corpus"
# Fallback only: 01_3's output, whose `split` column is the OLD sha1(filename) hash
# split. §3 will run on it but marks every result not_publishable.
LEGACY_PARQUET = DATA_ROOT / "pairs_sonar_filtered.parquet"

RUN_DIR = WORK_ROOT / RUN_NAME
SHARD_DIR = RUN_DIR / "shards"
RESULT_PATH = RUN_DIR / f"{RUN_NAME}_segments.parquet"
METRIC_PATH = RUN_DIR / f"{RUN_NAME}_metrics.parquet"

# ---- evaluation set ----------------------------------------------------------
EVAL_SPLIT = "test"          # C5: the test split is touched once, at the end.
                             # Set "dev" for anything that informs a decision.
# The full test split is ~14.7k pairs; at beam 5 that is hours per direction.
# None runs all of it -- the headline table in the paper must use None.
MAX_EVAL_SEGMENTS = 2000
SAMPLE_SEED = 1

# Segments outside this NLLB token range are excluded and counted, not truncated
# silently. Uses the precomputed mk_nllb_tokens / sq_nllb_tokens columns.
MIN_TOKENS = 3
MAX_TOKENS = 192

# ---- generation ---------------------------------------------------------------
NUM_BEAMS = 5
MAX_NEW_TOKENS = 256
NO_REPEAT_NGRAM_SIZE = None   # legal text legitimately repeats -- see the header
BATCH_SIZE = 32
PRECISION = "fp16"
SHARD_SIZE = 256              # one checkpoint every SHARD_SIZE segments per direction

# ---- metrics -------------------------------------------------------------------
CHRF_WORD_ORDER = 2           # chrF2++
RUN_COMET = True
COMET_MODEL = "Unbabel/wmt22-comet-da"
COMET_BATCH_SIZE = 32

print(RUN_NAME, "|", MODEL_NAME)
print("eval split:", EVAL_SPLIT, "| cap:", MAX_EVAL_SEGMENTS or "full split")
print("run dir:", RUN_DIR)

---
## 2. Chart styling


### Chart styling

The same palette as `01_1`–`01_3`, in the same slot order, so a colour means the same thing
across every notebook in the project.


In [ ]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # categorical, fixed order
SEQ_BLUE = "#2a78d6"
SEQ_LIGHT = "#cde2fb"
INK = "#0b0b0b"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
SURFACE = "#fcfcfb"
STATUS = {"good": "#0ca30c", "warning": "#fab219", "critical": "#d03b3b"}

plt.rcParams.update(
    {
        "figure.figsize": (11, 4.2),
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": INK_MUTED,
        "ytick.color": INK_MUTED,
        "text.color": INK,
        "legend.frameon": False,
        "lines.linewidth": 2.0,
        "font.size": 10,
    }
)


def thousands(ax, axis="y"):
    """Format an axis with thousands separators."""
    fmt = plt.FuncFormatter(lambda v, _: f"{v:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)
    return ax

---
## 3. Load the evaluation split

Two sources, and which one is used changes what the result may be called.

* **`data/processed/corpus/`** — the chronological, issue-disjoint corpus from
  `scripts/prepare_corpus.py`. Train is the oldest issues, dev the next, test the newest, and
  a cross-split MinHash sweep has removed evaluation segments that near-duplicate training
  text. This is the only source a publishable number may come from.
* **`pairs_sonar_filtered.parquet`** — `01_3`'s output, whose `split` column is still
  `sha1(filename)[:8] % 100`. Every year appears in all three splits and the formulaic text
  that the gazette repeats across issues sits on both sides of the boundary. Usable for a
  smoke test; `PUBLISHABLE` is set `False` and travels into the result file so a number from
  it can never be mistaken for the real thing later.


In [ ]:
def load_eval_split() -> tuple[pd.DataFrame, bool, str]:
    """(frame, publishable, provenance) for the configured EVAL_SPLIT."""
    if CORPUS_DIR.exists():
        table = pq.read_table(CORPUS_DIR, filters=[("split", "==", EVAL_SPLIT)])
        return table.to_pandas(), True, f"chronological corpus at {CORPUS_DIR}"

    if LEGACY_PARQUET.exists():
        frame = pd.read_parquet(LEGACY_PARQUET)
        frame = frame[frame["split"] == EVAL_SPLIT].copy()
        return frame, False, f"LEGACY hash split at {LEGACY_PARQUET}"

    raise FileNotFoundError(
        f"no corpus found.\n"
        f"  expected {CORPUS_DIR} (run scripts/prepare_corpus.py), or\n"
        f"  {LEGACY_PARQUET} (01_3's output) as a fallback"
    )


eval_df, PUBLISHABLE, PROVENANCE = load_eval_split()
print(f"source: {PROVENANCE}")
print(f"{len(eval_df):,} pairs in split {EVAL_SPLIT!r}")

if not PUBLISHABLE:
    print()
    print("!" * 78)
    print("  The split column is the OLD sha1(filename) hash split, not the chronological")
    print("  one. Gazette boilerplate repeats across issues and years, so train and test")
    print("  text overlap and every score below is inflated by memorisation.")
    print("  Results are written with publishable=False. Run scripts/prepare_corpus.py")
    print("  and re-run this notebook before quoting any number from it.")
    print("!" * 78)

In [ ]:
# --- token-length window ---------------------------------------------------------
# prepare_corpus.py precomputes these so no notebook ever re-tokenises the corpus.
if {"mk_nllb_tokens", "sq_nllb_tokens"} <= set(eval_df.columns):
    mk_tokens, sq_tokens = eval_df["mk_nllb_tokens"], eval_df["sq_nllb_tokens"]
    token_source = "precomputed mk_nllb_tokens / sq_nllb_tokens"
else:
    # Legacy fallback: approximate from characters so the notebook still runs.
    mk_tokens = (eval_df["mk_chars"] / 3.2).round().astype(int)
    sq_tokens = (eval_df["sq_chars"] / 3.6).round().astype(int)
    token_source = "APPROXIMATED from mk_chars / sq_chars -- run prepare_corpus.py"

in_window = (
    mk_tokens.between(MIN_TOKENS, MAX_TOKENS) & sq_tokens.between(MIN_TOKENS, MAX_TOKENS)
)
print(f"token lengths: {token_source}")
print(f"inside [{MIN_TOKENS}, {MAX_TOKENS}] tokens on both sides: "
      f"{int(in_window.sum()):,} of {len(eval_df):,} ({in_window.mean():.1%})")
print(f"excluded: {int((~in_window).sum()):,} "
      f"(too short {int((mk_tokens < MIN_TOKENS).sum() + (sq_tokens < MIN_TOKENS).sum()):,}, "
      f"too long {int((mk_tokens > MAX_TOKENS).sum() + (sq_tokens > MAX_TOKENS).sum()):,})")

evaluation = eval_df[in_window].copy()

# --- subsample, issue-stratified, fixed seed --------------------------------------
N_BEFORE_SAMPLE = len(evaluation)
if MAX_EVAL_SEGMENTS is not None and len(evaluation) > MAX_EVAL_SEGMENTS:
    evaluation = evaluation.sample(MAX_EVAL_SEGMENTS, random_state=SAMPLE_SEED).sort_index()
    print(f"\nsubsampled to {len(evaluation):,} of {N_BEFORE_SAMPLE:,} segments "
          f"(seed {SAMPLE_SEED}) -- set MAX_EVAL_SEGMENTS=None for the full split")

# A stable per-segment key. sq_component_id is unique across the corpus and survives
# a re-run, which is what makes §5 resumable.
evaluation["segment_uid"] = evaluation["sq_component_id"].astype(str)
assert evaluation["segment_uid"].is_unique, "segment_uid is not unique"

issue_column = "issue_key" if "issue_key" in evaluation.columns else "filename"
print(f"\nevaluating {len(evaluation):,} segments from "
      f"{evaluation[issue_column].nunique():,} issues, "
      f"years {evaluation['year'].min()}-{evaluation['year'].max()}")
print(evaluation[["mk_chars", "sq_chars"]].describe().round(1).to_string())

---
## 4. Model, tokenizer, and the `forced_bos_token_id` canary

Three checks, in order, before anything long runs.

1. **Both language codes exist in the tokenizer.** If either is missing the plan's "no
   vocabulary surgery" assumption is wrong and everything downstream is invalid.
2. **`forced_bos_token_id` resolves to the target language token**, and back again.
3. **Three real segments are generated and the script of the output is measured.** This is the
   check that matters: a wrong `forced_bos_token_id` produces fluent output in the wrong
   language, which passes every type check and every id assertion, and only shows up as a
   catastrophic BLEU after a full evaluation pass has been spent.


In [ ]:
TORCH_DTYPE = {"fp32": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}[PRECISION]

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=TORCH_DTYPE)
model.to(DEVICE).eval()
print(f"loaded {MODEL_NAME} in {time.time() - t0:,.1f}s | {PRECISION} | "
      f"{sum(p.numel() for p in model.parameters()) / 1e6:,.0f}M parameters")


def lang_token_id(code: str) -> int:
    """The tokenizer id of an NLLB language code, however this version exposes it."""
    mapping = getattr(tokenizer, "lang_code_to_id", None)
    if mapping and code in mapping:
        return int(mapping[code])
    token_id = tokenizer.convert_tokens_to_ids(code)
    if token_id is None or token_id == tokenizer.unk_token_id:
        raise RuntimeError(f"{MODEL_NAME} tokenizer has no language code {code!r}")
    return int(token_id)


# --- 1. both codes are present, no embedding resize needed -------------------------
for code in (MK, SQ):
    token_id = lang_token_id(code)
    assert tokenizer.convert_ids_to_tokens(token_id) == code, f"{code} does not round-trip"
    assert token_id < model.get_input_embeddings().num_embeddings, f"{code} is outside the embedding table"
    print(f"  {code:<10} id {token_id:>6}  round-trips, inside the embedding table")

# NLLB-200 ships an embedding table padded past the tokenizer: 256206 rows for
# 256204 tokens, the top 2 never addressed. That is stock, not vocabulary surgery.
# What has to hold is that every id the tokenizer can produce is in range.
n_embeddings = model.get_input_embeddings().num_embeddings
assert n_embeddings >= len(tokenizer), (
    f"embedding table ({n_embeddings}) is smaller than the tokenizer ({len(tokenizer)}) "
    f"-- token ids would index out of range"
)
print(f"  vocabulary intact: {len(tokenizer):,} tokens, {n_embeddings:,} embedding rows "
      f"({n_embeddings - len(tokenizer)} padding), no resize needed")

In [ ]:
CYRILLIC = re.compile(r"[\u0400-\u04FF]")
LATIN = re.compile(r"[A-Za-zÇçËë]")

SCRIPT_OF_LANG = {MK: "cyrillic", SQ: "latin"}


def script_share(text: str) -> dict:
    """Share of letters that are Cyrillic vs Latin. Punctuation and digits ignored."""
    cyr, lat = len(CYRILLIC.findall(text)), len(LATIN.findall(text))
    total = cyr + lat
    return {"cyrillic": cyr / total if total else 0.0, "latin": lat / total if total else 0.0}


@torch.inference_mode()
def translate(texts: list[str], src_lang: str, tgt_lang: str, batch_size: int = None) -> list[str]:
    """Translate `texts` from src_lang to tgt_lang, forcing the target language token."""
    tokenizer.src_lang = src_lang
    forced_bos = lang_token_id(tgt_lang)
    out: list[str] = []
    step = batch_size or BATCH_SIZE
    for start in range(0, len(texts), step):
        chunk = texts[start : start + step]
        encoded = tokenizer(
            chunk, return_tensors="pt", padding=True, truncation=True, max_length=MAX_TOKENS
        ).to(DEVICE)
        kwargs = dict(
            forced_bos_token_id=forced_bos,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            early_stopping=True,
        )
        if NO_REPEAT_NGRAM_SIZE:
            kwargs["no_repeat_ngram_size"] = NO_REPEAT_NGRAM_SIZE
        generated = model.generate(**encoded, **kwargs)
        out.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return out


# --- 2 & 3. the canary ---------------------------------------------------------------
canary = evaluation.head(3)
print("forced_bos_token_id canary\n" + "-" * 78)
for name, src_lang, tgt_lang in DIRECTIONS:
    source_column = "mk_text" if src_lang == MK else "sq_text"
    forced_bos = lang_token_id(tgt_lang)
    assert tokenizer.convert_ids_to_tokens(forced_bos) == tgt_lang

    hypotheses = translate(canary[source_column].tolist(), src_lang, tgt_lang, batch_size=3)
    expected = SCRIPT_OF_LANG[tgt_lang]
    shares = [script_share(h)[expected] for h in hypotheses]
    worst = min(shares)

    verdict = "PASS" if worst >= 0.80 else "FAIL"
    print(f"{verdict}  {name}: forced_bos_token_id={forced_bos} ({tgt_lang}), "
          f"output is {worst:.0%} {expected} at worst")
    for source, hypothesis in zip(canary[source_column], hypotheses):
        print(f"      {source[:64]!r}")
        print(f"   -> {hypothesis[:64]!r}")
    assert worst >= 0.80, (
        f"{name}: generated text is not {expected} -- forced_bos_token_id is wrong, "
        f"or src_lang was not set. Do not let §5 run."
    )
print("-" * 78)
print("both directions generate in the target script; §5 may run")

---
## 5. Generate — sharded, checkpointed, resumable

Colab sessions die. Generation is the expensive part of this notebook, so it is written in
shards of `SHARD_SIZE` segments per direction, each shard a Parquet file on Drive named by
its direction and index. A re-run reads the shards that already exist and generates only the
missing ones, so an interrupted pass resumes instead of restarting.

The shard is keyed by `segment_uid`, not by position, so it stays valid if `MAX_EVAL_SEGMENTS`
or the sample seed changes — a stale shard is detected and regenerated rather than silently
joined onto the wrong segment.


In [ ]:
SHARD_DIR.mkdir(parents=True, exist_ok=True)

SHARD_SCHEMA = pa.schema([
    ("segment_uid", pa.string()),
    ("direction", pa.string()),
    ("source", pa.string()),
    ("reference", pa.string()),
    ("hypothesis", pa.string()),
    ("gen_seconds", pa.float64()),
])


def shard_path(direction: str, index: int) -> Path:
    return SHARD_DIR / f"{direction}.{index:05d}.parquet"


def generate_direction(name: str, src_lang: str, tgt_lang: str) -> pd.DataFrame:
    """All shards for one direction, generating only what is missing."""
    source_column = "mk_text" if src_lang == MK else "sq_text"
    target_column = "sq_text" if tgt_lang == SQ else "mk_text"

    uids = evaluation["segment_uid"].to_numpy()
    sources = evaluation[source_column].to_numpy()
    references = evaluation[target_column].to_numpy()
    n_shards = int(np.ceil(len(evaluation) / SHARD_SIZE))

    done, produced = 0, 0
    for index in range(n_shards):
        lo, hi = index * SHARD_SIZE, min((index + 1) * SHARD_SIZE, len(evaluation))
        path = shard_path(name, index)
        expected_uids = list(uids[lo:hi])

        if path.exists():
            existing = pq.read_table(path)
            if list(existing.column("segment_uid").to_pylist()) == expected_uids:
                done += 1
                continue
            print(f"  {path.name}: stale (different segments) -- regenerating")

        started = time.time()
        hypotheses = translate(list(sources[lo:hi]), src_lang, tgt_lang)
        elapsed = time.time() - started

        pq.write_table(
            pa.Table.from_arrays(
                [
                    pa.array(expected_uids, pa.string()),
                    pa.array([name] * (hi - lo), pa.string()),
                    pa.array(list(sources[lo:hi]), pa.string()),
                    pa.array(list(references[lo:hi]), pa.string()),
                    pa.array(hypotheses, pa.string()),
                    pa.array([elapsed / (hi - lo)] * (hi - lo), pa.float64()),
                ],
                schema=SHARD_SCHEMA,
            ),
            path,
            compression="zstd",
        )
        produced += 1
        rate = (hi - lo) / elapsed
        remaining = (n_shards - index - 1) * SHARD_SIZE / rate
        print(f"  {name} shard {index + 1:>3}/{n_shards}  {elapsed:6.1f}s  "
              f"{rate:5.2f} seg/s  ~{remaining / 60:5.1f} min left")

    print(f"{name}: {done} shards reused, {produced} generated")
    return pa.concat_tables([pq.read_table(shard_path(name, i)) for i in range(n_shards)]).to_pandas()


frames = []
for name, src_lang, tgt_lang in DIRECTIONS:
    print(f"\n=== {name} ({src_lang} -> {tgt_lang}) ===")
    frames.append(generate_direction(name, src_lang, tgt_lang))

generated = pd.concat(frames, ignore_index=True)
assert len(generated) == len(evaluation) * len(DIRECTIONS), "a shard is missing"
assert not generated["hypothesis"].isna().any(), "a hypothesis is null"
print(f"\n{len(generated):,} translations "
      f"({len(evaluation):,} segments x {len(DIRECTIONS)} directions)")
print(f"mean {generated['gen_seconds'].mean():.3f}s per segment")
generated.head(3)

In [ ]:
# An empty hypothesis is not a translation failure BLEU can express usefully, and a
# handful of them will drag a corpus score down without saying why. Counted, kept.
empty = generated["hypothesis"].str.strip().eq("")
copy_through = (
    generated["hypothesis"].str.strip().str.casefold()
    == generated["source"].str.strip().str.casefold()
)
report_rows = generated.assign(empty=empty, copy_through=copy_through).groupby("direction").agg(
    segments=("hypothesis", "size"),
    empty=("empty", "sum"),
    copy_through=("copy_through", "sum"),
    mean_hyp_chars=("hypothesis", lambda s: s.str.len().mean()),
    mean_ref_chars=("reference", lambda s: s.str.len().mean()),
)
report_rows["length_ratio"] = (report_rows["mean_hyp_chars"] / report_rows["mean_ref_chars"]).round(3)
report_rows.round(1)

---
## 6. General quality: SacreBLEU, chrF2++, COMET

Three metrics, per direction, never averaged across directions.

* **SacreBLEU** with its full signature recorded. The signature is what makes the number
  comparable to anyone else's, and it goes into the result file rather than into a comment.
* **chrF2++** (`word_order=2`). More informative than BLEU for both of these languages:
  Macedonian and Albanian are morphologically rich, and a character-n-gram metric does not
  punish a correct translation for inflecting a word differently from the reference.
* **COMET** `Unbabel/wmt22-comet-da`. Reported with an explicit caveat — COMET's training
  data does not cover mk↔sq, and it has not been validated on this pair. It is here because
  a neural metric catches adequacy failures that surface metrics miss, not because its
  absolute values mean anything on this language pair.


In [ ]:
def corpus_metrics(frame: pd.DataFrame) -> dict:
    """SacreBLEU and chrF2++ for one direction, with signatures."""
    hypotheses = frame["hypothesis"].tolist()
    references = [frame["reference"].tolist()]

    bleu_scorer = sacrebleu.BLEU()
    chrf_scorer = sacrebleu.CHRF(word_order=CHRF_WORD_ORDER)
    bleu = bleu_scorer.corpus_score(hypotheses, references)
    chrf = chrf_scorer.corpus_score(hypotheses, references)

    return {
        "bleu": bleu.score,
        "bleu_signature": str(bleu_scorer.get_signature()),
        "chrf2pp": chrf.score,
        "chrf_signature": str(chrf_scorer.get_signature()),
        "segments": len(frame),
    }


surface = {name: corpus_metrics(group) for name, group in generated.groupby("direction")}
surface_table = pd.DataFrame(surface).T
print(surface_table[["segments", "bleu", "chrf2pp"]].round(2).to_string())
print()
for name, values in surface.items():
    print(f"{name} BLEU   {values['bleu_signature']}")
    print(f"{name} chrF2++ {values['chrf_signature']}")

In [ ]:
comet_scores = {name: np.nan for name, _, _ in DIRECTIONS}
comet_segment = pd.Series(np.nan, index=generated.index, dtype="float64")

if RUN_COMET:
    try:
        _ensure("unbabel-comet", "comet")
        from comet import download_model, load_from_checkpoint

        checkpoint = download_model(COMET_MODEL)
        comet_model = load_from_checkpoint(checkpoint)
        for name, group in generated.groupby("direction"):
            payload = [
                {"src": s, "mt": h, "ref": r}
                for s, h, r in zip(group["source"], group["hypothesis"], group["reference"])
            ]
            output = comet_model.predict(
                payload, batch_size=COMET_BATCH_SIZE,
                gpus=1 if DEVICE == "cuda" else 0, progress_bar=True,
            )
            comet_scores[name] = float(output.system_score)
            comet_segment.loc[group.index] = output.scores
        del comet_model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        print("\nCOMET system scores:", {k: round(v, 4) for k, v in comet_scores.items()})
    except Exception as exc:
        print(f"COMET skipped: {type(exc).__name__}: {exc}")
        print("The surface metrics above are unaffected.")
else:
    print("COMET disabled (RUN_COMET = False)")

print("\nCAVEAT for the paper: wmt22-comet-da has not been validated on mkd_Cyrl<->als_Latn.")
print("Report it as indicative and never as the sole basis for a claim.")

---
## 7. Pattern fidelity

BLEU cannot tell you that the model wrote «член 7» where the source said «член 5». A dropped
or altered article number is a hard failure in a legal translation regardless of how fluent
the sentence around it is, and it is the failure mode this corpus is most likely to produce.

Six families, all regex-extractable, so this runs over the whole evaluation set with no
annotation. Each family extracts a **set of values** — not words — from the source and from
the hypothesis, and scores a segment as a hit when the two sets are equal. Values are
language-independent, which is what makes a cross-lingual set comparison meaningful at all.

| Family | Source-side cue | What is compared |
|---|---|---|
| `article` | `член` / `членот` · `nen(i\|in\|it)` | the article number |
| `paragraph` | `став` / `ставот` · `paragraf(i\|in\|it)` | the paragraph number |
| `date` | `11.01.2024`, `11/1/2024` | normalised (day, month, year) |
| `money` | `денари` · `denarë` · `MKD` · `EUR` | normalised amount + currency |
| `percent` | `15%`, `15 %` | normalised amount |
| `numeral` | any bare number | every normalised number in the segment |

**Numbers are normalised before comparison.** Both languages write `1.000,50` where English
writes `1,000.50`, and a naive string match would score a correct translation as a failure
over a separator. The rule: a single separator with exactly three digits after it and
nothing else is a thousands separator, anything else is a decimal point. It is a heuristic
and it is wrong on `1.500` meaning one-point-five — rare enough in gazette text to accept,
and documented here rather than hidden.


In [ ]:
# A number, and only a number. A separator must be followed immediately by digits, so
# an enumeration like «членови 5, 7 и 9» yields {5, 7, 9} instead of one long number
# glued across the commas. A space or NBSP counts only before exactly three digits.
NUMBER = r"\d+(?:[.,]\d+|[\u00a0 ]\d{3}(?!\d))*"

PATTERNS = {
    "article": re.compile(rf"(?:член(?:от|овите|ови)?|nen(?:i|in|it|e|et)?)\s*({NUMBER})", re.I),
    "paragraph": re.compile(rf"(?:став(?:от|овите)?|paragraf(?:i|in|it|e)?)\s*({NUMBER})", re.I),
    "date": re.compile(r"\b(\d{1,2})\s*[.\-/]\s*(\d{1,2})\s*[.\-/]\s*(\d{4})\b"),
    "money": re.compile(
        rf"({NUMBER})\s*(денар[иа]?|denar[ëe]?v?|MKD|EUR|евр[оаи]|euro?)\b", re.I
    ),
    "percent": re.compile(rf"({NUMBER})\s*%"),
    "numeral": re.compile(NUMBER),
}

CURRENCY_CLASS = {
    "денар": "MKD", "денари": "MKD", "денара": "MKD", "mkd": "MKD",
    "denar": "MKD", "denare": "MKD", "denarë": "MKD", "denarëv": "MKD",
    "eur": "EUR", "евро": "EUR", "евра": "EUR", "еври": "EUR", "euro": "EUR", "eur o": "EUR",
}


def normalise_number(raw: str) -> str:
    """`1.000,50` and `1 000,50` -> `1000.5`. See the heading for the separator rule."""
    text = raw.replace("\u00a0", "").replace(" ", "").strip(".,")
    if not text:
        return ""
    last_dot, last_comma = text.rfind("."), text.rfind(",")
    if last_dot >= 0 and last_comma >= 0:
        decimal_at = max(last_dot, last_comma)
        integer = re.sub(r"[.,]", "", text[:decimal_at])
        fraction = re.sub(r"[^0-9]", "", text[decimal_at + 1 :])
    elif last_dot >= 0 or last_comma >= 0:
        at = max(last_dot, last_comma)
        tail = text[at + 1 :]
        if len(tail) == 3 and text.count(text[at]) >= 1 and len(text[:at].replace(".", "").replace(",", "")) <= 3:
            # a single separator with exactly three digits after it: thousands
            integer, fraction = re.sub(r"[.,]", "", text), ""
        elif len(tail) == 3 and text.count(".") + text.count(",") > 1:
            integer, fraction = re.sub(r"[.,]", "", text), ""
        else:
            integer, fraction = re.sub(r"[^0-9]", "", text[:at]), re.sub(r"[^0-9]", "", tail)
    else:
        integer, fraction = re.sub(r"[^0-9]", "", text), ""
    integer = integer.lstrip("0") or "0"
    fraction = fraction.rstrip("0")
    return f"{integer}.{fraction}" if fraction else integer


def extract(family: str, text: str) -> frozenset:
    """The set of comparable values of `family` present in `text`."""
    pattern = PATTERNS[family]
    if family == "date":
        return frozenset(
            f"{int(d)}-{int(m)}-{int(y)}" for d, m, y in pattern.findall(text)
        )
    if family == "money":
        return frozenset(
            f"{normalise_number(amount)} {CURRENCY_CLASS.get(unit.casefold(), unit.upper())}"
            for amount, unit in pattern.findall(text)
        )
    return frozenset(normalise_number(match) for match in pattern.findall(text)) - {""}


_t = "Согласно член 5 став 2, износот од 1.250,50 денари (15%) на ден 11.01.2024 година."
assert extract("article", _t) == {"5"}, extract("article", _t)
assert extract("paragraph", _t) == {"2"}
assert extract("money", _t) == {"1250.5 MKD"}, extract("money", _t)
assert extract("percent", _t) == {"15"}
assert extract("date", _t) == {"11-1-2024"}
assert extract("article", "Sipas nenit 5 paragrafi 2") == {"5"}
assert extract("paragraph", "Sipas nenit 5 paragrafi 2") == {"2"}
assert normalise_number("1.000") == "1000" and normalise_number("1.000,50") == "1000.5"
assert normalise_number("15,5") == "15.5" and normalise_number("2024") == "2024"
# an enumeration must not collapse into one number -- see the NUMBER comment above
assert extract("numeral", "член 5, 1.250,50 денари, 15%") == {"5", "1250.5", "15"}
assert extract("numeral", "членови 5, 7 и 9") == {"5", "7", "9"}
del _t
print("pattern extractors pass their unit checks")

In [ ]:
def score_patterns(frame: pd.DataFrame) -> pd.DataFrame:
    """Per family: how often the hypothesis carries exactly the source's values."""
    records = []
    for family in PATTERNS:
        source_sets = [extract(family, t) for t in frame["source"]]
        hypothesis_sets = [extract(family, t) for t in frame["hypothesis"]]

        fired = [i for i, s in enumerate(source_sets) if s]
        if not fired:
            records.append({"family": family, "segments_with_pattern": 0})
            continue

        exact = sum(source_sets[i] == hypothesis_sets[i] for i in fired)
        recovered = sum(len(source_sets[i] & hypothesis_sets[i]) for i in fired)
        expected = sum(len(source_sets[i]) for i in fired)
        produced = sum(len(hypothesis_sets[i]) for i in fired)

        records.append({
            "family": family,
            "segments_with_pattern": len(fired),
            "pct_of_segments": len(fired) / len(frame),
            "exact_set_match": exact / len(fired),
            "value_recall": recovered / expected if expected else np.nan,
            "value_precision": recovered / produced if produced else np.nan,
            "values_expected": expected,
        })
    return pd.DataFrame(records).set_index("family")


pattern_tables = {name: score_patterns(group) for name, group in generated.groupby("direction")}
for name, table in pattern_tables.items():
    print(f"\n=== {name} ===")
    print(table.round(3).to_string())

In [ ]:
families = [f for f in PATTERNS if pattern_tables[DIRECTIONS[0][0]].loc[f, "segments_with_pattern"]]
fig, ax = plt.subplots(figsize=(11, 3.8))
width = 0.38
positions = np.arange(len(families))
for offset, (name, _, _) in enumerate(DIRECTIONS):
    values = [pattern_tables[name].loc[f, "exact_set_match"] for f in families]
    bars = ax.bar(positions + (offset - 0.5) * width, values, width,
                  color=SERIES[offset], label=name)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.0%}",
                ha="center", va="bottom", fontsize=8, color=INK)
ax.set_xticks(positions)
ax.set_xticklabels(families)
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"Pattern fidelity, {MODEL_NAME.split('/')[-1]} zero-shot "
             f"(exact set match, source vs hypothesis)")
ax.set_ylabel("segments where the value set matches exactly")
ax.grid(axis="x", visible=False)
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Results, per direction

One table. MK→SQ and SQ→MK on separate rows, never averaged — under adaptation these two
move independently, and the whole point of reporting M0 is to have an honest floor for each
of them separately.


In [ ]:
rows = []
for name, _, _ in DIRECTIONS:
    row = {
        "direction": name,
        "segments": int(surface[name]["segments"]),
        "bleu": surface[name]["bleu"],
        "chrf2pp": surface[name]["chrf2pp"],
        "comet": comet_scores[name],
    }
    table = pattern_tables[name]
    for family in PATTERNS:
        if table.loc[family, "segments_with_pattern"]:
            row[f"{family}_exact"] = table.loc[family, "exact_set_match"]
            row[f"{family}_recall"] = table.loc[family, "value_recall"]
    rows.append(row)

results = pd.DataFrame(rows).set_index("direction")
display(results.round(4))

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
names = [d[0] for d in DIRECTIONS]
left.bar(names, [surface[n]["bleu"] for n in names], color=SERIES[0], width=0.5)
for x, n in enumerate(names):
    left.text(x, surface[n]["bleu"], f"  {surface[n]['bleu']:.2f}", ha="center", va="bottom", color=INK)
left.set_title("SacreBLEU")
left.set_ylabel("BLEU")
left.grid(axis="x", visible=False)

right.bar(names, [surface[n]["chrf2pp"] for n in names], color=SERIES[2], width=0.5)
for x, n in enumerate(names):
    right.text(x, surface[n]["chrf2pp"], f"  {surface[n]['chrf2pp']:.2f}", ha="center", va="bottom", color=INK)
right.set_title("chrF2++")
right.set_ylabel("chrF2++")
right.grid(axis="x", visible=False)
fig.suptitle(f"{MODEL_NAME.split('/')[-1]} zero-shot on the Gazette {EVAL_SPLIT} split",
             fontweight="bold", color=INK)
plt.tight_layout()
plt.show()

---
## 9. Write and report

Two Parquet files, explicit schemas, ZSTD, never CSV:

* **`{RUN_NAME}_segments.parquet`** — one row per (segment, direction): the source, the
  reference, the hypothesis and its per-segment COMET score. This is what a later error
  analysis and the manually verified legal evaluation set are built from.
* **`{RUN_NAME}_metrics.parquet`** — one row per direction, carrying the scores, the full
  SacreBLEU and chrF signatures, and the provenance needed to decide whether the number may
  be quoted: model, precision, beam width, split, sample size, seed, and `publishable`.

A number that does not trace to one of these files does not go in the paper.


In [ ]:
SEGMENT_SCHEMA = pa.schema([
    ("run_name", pa.string()),
    ("segment_uid", pa.string()),
    ("direction", pa.string()),
    ("source", pa.string()),
    ("reference", pa.string()),
    ("hypothesis", pa.string()),
    ("comet_segment", pa.float64()),
    ("gen_seconds", pa.float64()),
])

METRIC_SCHEMA = pa.schema([
    ("run_name", pa.string()),
    ("arm", pa.string()),
    ("strategy", pa.string()),
    ("seed", pa.int32()),
    ("model_name", pa.string()),
    ("precision", pa.string()),
    ("num_beams", pa.int32()),
    ("max_new_tokens", pa.int32()),
    ("direction", pa.string()),
    ("eval_split", pa.string()),
    ("eval_source", pa.string()),
    ("publishable", pa.bool_()),
    ("segments", pa.int32()),
    ("segments_available", pa.int32()),
    ("sample_seed", pa.int32()),
    ("bleu", pa.float64()),
    ("bleu_signature", pa.string()),
    ("chrf2pp", pa.float64()),
    ("chrf_signature", pa.string()),
    ("comet", pa.float64()),
    ("comet_model", pa.string()),
    ("pattern_scores", pa.string()),      # the §7 table as JSON, one blob per direction
    ("generated_utc", pa.string()),
])

RUN_DIR.mkdir(parents=True, exist_ok=True)

segments_table = pa.Table.from_arrays([
    pa.array([RUN_NAME] * len(generated), pa.string()),
    pa.array(generated["segment_uid"].tolist(), pa.string()),
    pa.array(generated["direction"].tolist(), pa.string()),
    pa.array(generated["source"].tolist(), pa.string()),
    pa.array(generated["reference"].tolist(), pa.string()),
    pa.array(generated["hypothesis"].tolist(), pa.string()),
    pa.array(comet_segment.reindex(generated.index).tolist(), pa.float64()),
    pa.array(generated["gen_seconds"].tolist(), pa.float64()),
], schema=SEGMENT_SCHEMA)
pq.write_table(segments_table, RESULT_PATH, compression="zstd")

stamp = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
metric_rows = []
for name, _, _ in DIRECTIONS:
    metric_rows.append({
        "run_name": RUN_NAME, "arm": ARM, "strategy": STRATEGY, "seed": SEED,
        "model_name": MODEL_NAME, "precision": PRECISION,
        "num_beams": NUM_BEAMS, "max_new_tokens": MAX_NEW_TOKENS,
        "direction": name, "eval_split": EVAL_SPLIT, "eval_source": PROVENANCE,
        "publishable": bool(PUBLISHABLE),
        "segments": int(surface[name]["segments"]), "segments_available": int(N_BEFORE_SAMPLE),
        "sample_seed": SAMPLE_SEED,
        "bleu": float(surface[name]["bleu"]), "bleu_signature": surface[name]["bleu_signature"],
        "chrf2pp": float(surface[name]["chrf2pp"]), "chrf_signature": surface[name]["chrf_signature"],
        "comet": float(comet_scores[name]) if comet_scores[name] == comet_scores[name] else None,
        "comet_model": COMET_MODEL if RUN_COMET else None,
        "pattern_scores": pattern_tables[name].round(6).to_json(orient="index"),
        "generated_utc": stamp,
    })
metrics_table = pa.Table.from_pylist(metric_rows, schema=METRIC_SCHEMA)
pq.write_table(metrics_table, METRIC_PATH, compression="zstd")

# round trip: re-read both and confirm they are what we think they are
back_segments = pq.read_table(RESULT_PATH)
back_metrics = pq.read_table(METRIC_PATH)
assert back_segments.num_rows == len(generated), "segment row count changed on the round trip"
assert back_metrics.num_rows == len(DIRECTIONS), "metric row count changed on the round trip"
assert back_segments.schema.equals(SEGMENT_SCHEMA), "segment schema changed on the round trip"
print(f"wrote {RESULT_PATH}  ({RESULT_PATH.stat().st_size / 1e6:,.1f} MB)")
print(f"wrote {METRIC_PATH}  ({METRIC_PATH.stat().st_size / 1e3:,.1f} KB)")
print(f"round trip OK: {back_segments.num_rows:,} segments, {back_metrics.num_rows} metric rows")

In [ ]:
provenance = {
    "run_name": RUN_NAME,
    "generated_utc": stamp,
    "model_name": MODEL_NAME,
    "precision": PRECISION,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "eval_split": EVAL_SPLIT,
    "eval_source": PROVENANCE,
    "publishable": bool(PUBLISHABLE),
    "segments_evaluated": int(len(evaluation)),
    "segments_available": int(N_BEFORE_SAMPLE),
    "sample_seed": SAMPLE_SEED,
    "generation": {
        "num_beams": NUM_BEAMS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
        "batch_size": BATCH_SIZE,
        "min_tokens": MIN_TOKENS,
        "max_tokens": MAX_TOKENS,
    },
    "versions": {
        "torch": torch.__version__,
        "transformers": __import__("transformers").__version__,
        "sacrebleu": sacrebleu.__version__,
        "pyarrow": pa.__version__,
        "pandas": pd.__version__,
    },
}
(RUN_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2, ensure_ascii=False))
print(json.dumps(provenance, indent=2, ensure_ascii=False))

In [ ]:
banner = "" if PUBLISHABLE else """
!!! NOT PUBLISHABLE -- the evaluation split came from the legacy sha1(filename) hash
!!! split, which leaks gazette boilerplate between train and test. Run
!!! scripts/prepare_corpus.py and re-run this notebook.
"""

report = f"""
СЛУЖБЕН ВЕСНИК -- M0 ZERO-SHOT, {MODEL_NAME}
======================================================================={banner}
run                                   {RUN_NAME:>32}
evaluation split                      {EVAL_SPLIT:>32}
source                                {PROVENANCE[-32:]:>32}
segments evaluated                    {len(evaluation):>32,}
  of available in the split           {N_BEFORE_SAMPLE:>32,}
issues represented                    {evaluation[issue_column].nunique():>32,}
years                                 {f"{evaluation['year'].min()}-{evaluation['year'].max()}":>32}
precision / beams                     {f"{PRECISION} / {NUM_BEAMS}":>32}

GENERAL QUALITY                       {"  ".join(f"{d:>10}" for d in results.index)}
  SacreBLEU                           {"  ".join(f"{results.loc[d, 'bleu']:>10.2f}" for d in results.index)}
  chrF2++                             {"  ".join(f"{results.loc[d, 'chrf2pp']:>10.2f}" for d in results.index)}
  COMET                               {"  ".join((f"{results.loc[d, 'comet']:>10.4f}" if pd.notna(results.loc[d, 'comet']) else f"{'n/a':>10}") for d in results.index)}

PATTERN FIDELITY (exact set match)    {"  ".join(f"{d:>10}" for d in results.index)}
""" + "\n".join(
    f"  {family:<36}" + "  ".join(
        (f"{results.loc[d, f'{family}_exact']:>10.1%}" if f"{family}_exact" in results.columns
         and pd.notna(results.loc[d, f"{family}_exact"]) else f"{'n/a':>10}")
        for d in results.index
    )
    for family in PATTERNS
) + f"""

SACREBLEU SIGNATURES
{chr(10).join(f"  {d}: {surface[d]['bleu_signature']}" for d in results.index)}

CAVEATS
  COMET (wmt22-comet-da) is not validated on mkd_Cyrl<->als_Latn. Indicative only.
  M0 is a floor: NLLB-200 saw both languages but no Macedonian legal gazette.
  Directions are reported separately and must not be averaged.

WRITTEN TO
  {RESULT_PATH}
  {METRIC_PATH}
=======================================================================
"""
print(report)
(RUN_DIR / "report.txt").write_text(report)

### Reading the output

```python
metrics = pd.read_parquet("{RUN_NAME}_metrics.parquet")
metrics[metrics.publishable][["direction", "bleu", "chrf2pp", "comet"]]

segments = pd.read_parquet("{RUN_NAME}_segments.parquet")

# the segments where an article number was lost -- the first place to look
import json
worst = segments[segments.direction == "mk2sq"].nsmallest(20, "comet_segment")
```

The next notebook in the sequence fine-tunes on D0 and writes an `M1` run with the same two
schemas, so the comparison is a concat and a groupby rather than a transcription.
